# LeetCode 489 — Robot Room Cleaner bằng Python

- Link: https://leetcode.com/problems/robot-room-cleaner/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: DFS + Backtracking, Tọa độ tương đối, Mô phỏng tương tác qua interface

> **Ghi chú Premium:** Đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết phổ biến về bài toán (interface `Robot`, ví dụ, ràng buộc). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** câu chữ và ràng buộc chính xác trên LeetCode. Vì LeetCode không cho ta lớp `Robot` thật, notebook **tự dựng lớp mô phỏng** `RobotSim` để chạy và kiểm thử.

Skeleton trên LeetCode (Python3):

```python
# """
# This is the robot's control interface.
# You should not implement it, or speculate about its implementation
# """
#class Robot:
#    def move(self):
#        """
#        Returns true if the cell in front is open and robot moves into the cell.
#        Returns false if the cell in front is blocked and robot stays in the current cell.
#        :rtype bool
#        """
#    def turnLeft(self):
#        """Robot will stay in the same cell after calling turnLeft/turnRight. Each turn will be 90 degrees."""
#    def turnRight(self):
#    def clean(self):
#        """Clean the current cell."""

class Solution:
    def cleanRoom(self, robot):
        """
        :type robot: Robot
        :rtype: None
        """
```

## 1. Mục tiêu học

1. Làm quen với bài **tương tác (interactive)**: ta không thấy bản đồ, chỉ điều khiển robot qua 4 lệnh `move`, `turnLeft`, `turnRight`, `clean`.
2. Dùng **tọa độ tương đối**: coi ô xuất phát là `(0, 0)`, hướng ban đầu là "lên", tự theo dõi vị trí và hướng.
3. Nắm kỹ thuật **DFS + backtracking vật lý**: sau khi khám phá xong một nhánh, robot phải **thật sự đi lùi** về ô cũ và quay lại đúng hướng.
4. So sánh 3 cách: DFS "quay về hướng tuyệt đối", DFS **xoắn ốc** (spiral, lời giải chuẩn), DFS **lặp với stack** (tránh giới hạn đệ quy).
5. Tự dựng **lớp mô phỏng** để kiểm thử bài interactive — kỹ năng rất hữu ích khi luyện các bài có interface.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có một căn phòng dạng lưới `m × n`, mỗi ô là `1` (trống, đi được) hoặc `0` (tường). Một robot hút bụi đặt ở một ô trống, **hướng lên trên**. Bạn **không biết** bản đồ, không biết vị trí robot. Bạn chỉ có thể gọi:

- `robot.move()` → `True` nếu ô phía trước trống và robot tiến vào; `False` nếu phía trước là tường/biên (robot đứng yên).
- `robot.turnLeft()`, `robot.turnRight()` → quay 90° tại chỗ.
- `robot.clean()` → lau ô hiện tại.

Hãy viết `cleanRoom(robot)` để robot **lau mọi ô trống có thể đi tới** từ vị trí ban đầu. Mọi ô ngoài lưới coi như tường.

**Ví dụ 1**

```text
Input:
room = [
  [1,1,1,1,1,0,1,1],
  [1,1,1,1,1,0,1,1],
  [1,0,1,1,1,1,1,1],
  [0,0,0,1,0,0,0,0],
  [1,1,1,1,1,1,1,1]
],
row = 1, col = 3
Output: Robot cleaned all rooms.
Giải thích: robot bắt đầu ở (1, 3), hướng lên. Mọi ô trống liên thông đều phải được lau.
```

**Ví dụ 2**

```text
Input: room = [[1]], row = 0, col = 0
Output: Robot cleaned all rooms.
```

**Ràng buộc (theo hiểu biết, nên đối chiếu)**

- `m == room.length`, `n == room[i].length`
- `1 <= m <= 100`, `1 <= n <= 200`
- `room[i][j]` là `0` hoặc `1`; `room[row][col] == 1`
- Mọi ô trống liên thông với ô xuất phát (nếu không liên thông thì robot đương nhiên không tới được).

Input `room, row, col` chỉ dùng để **khởi tạo** robot trong hệ thống chấm; code của bạn không được đọc trực tiếp.

Vì không có lớp `Robot` thật, ta dựng **`RobotSim`** — mô phỏng đúng 4 lệnh, đồng thời đếm số lệnh và ghi lại các ô đã lau để kiểm thử.

In [1]:
class RobotSim:
    """Mô phỏng robot. Hướng: 0=lên, 1=phải, 2=xuống, 3=trái (theo chiều kim đồng hồ)."""
    DR = [-1, 0, 1, 0]
    DC = [0, 1, 0, -1]

    def __init__(self, room, row, col, facing=0, log=False):
        self.room = room
        self.r, self.c = row, col
        self.d = facing
        self.cleaned = set()
        self.moves = self.turns = 0
        self.log = log

    def move(self):
        nr, nc = self.r + self.DR[self.d], self.c + self.DC[self.d]
        self.moves += 1
        if 0 <= nr < len(self.room) and 0 <= nc < len(self.room[0]) and self.room[nr][nc] == 1:
            self.r, self.c = nr, nc
            if self.log: print(f"   move -> ({nr},{nc})")
            return True
        if self.log: print("   move -> bị chặn")
        return False

    def turnLeft(self):
        self.d = (self.d + 3) % 4; self.turns += 1

    def turnRight(self):
        self.d = (self.d + 1) % 4; self.turns += 1

    def clean(self):
        self.cleaned.add((self.r, self.c))
        if self.log: print(f"   clean ({self.r},{self.c})")

room1 = [
    [1, 1, 1, 1, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 0, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1],
]
rb = RobotSim(room1, 1, 3)
print("move lên:", rb.move(), "vị trí thật:", (rb.r, rb.c))
print("move lên tiếp:", rb.move(), "(ra ngoài lưới -> False)")
rb.clean(); print("đã lau:", rb.cleaned)

move lên: True vị trí thật: (0, 3)
move lên tiếp: False (ra ngoài lưới -> False)
đã lau: {(0, 3)}


## 3. Kiến thức nền

### 3.1. Mã hóa hướng bằng số và mảng `dr`, `dc`

Đánh số hướng **theo chiều kim đồng hồ**: `0 = lên`, `1 = phải`, `2 = xuống`, `3 = trái`. Bước đi theo hướng `d`: `(r + dr[d], c + dc[d])` với `dr = [-1, 0, 1, 0]`, `dc = [0, 1, 0, -1]`.

- Quay phải: `d = (d + 1) % 4`.
- Quay trái: `d = (d + 3) % 4` (tương đương `d - 1` nhưng không âm).
- Quay 180°: `d = (d + 2) % 4`.

**Điều cực kỳ quan trọng:** thứ tự trong `dr/dc` phải **khớp** với chiều quay `turnRight`. Nếu liệt kê theo chiều kim đồng hồ thì `turnRight` tương ứng `d + 1`.

In [2]:
dr = [-1, 0, 1, 0]
dc = [0, 1, 0, -1]
names = ["lên", "phải", "xuống", "trái"]
d = 0
for _ in range(5):
    print(f"hướng {d} ({names[d]}): bước = ({dr[d]}, {dc[d]})")
    d = (d + 1) % 4
print("quay trái từ 0 ->", (0 + 3) % 4, names[(0 + 3) % 4])

hướng 0 (lên): bước = (-1, 0)
hướng 1 (phải): bước = (0, 1)
hướng 2 (xuống): bước = (1, 0)
hướng 3 (trái): bước = (0, -1)
hướng 0 (lên): bước = (-1, 0)
quay trái từ 0 -> 3 trái


### 3.2. Tọa độ tương đối

Robot không biết mình ở đâu trong lưới thật. Nhưng ta có thể **tự đặt** hệ tọa độ: ô xuất phát là `(0, 0)`, hướng xuất phát là `0` ("lên"). Mỗi lần `move()` thành công theo hướng `d`, ta cộng `(dr[d], dc[d])`. Tọa độ có thể âm — không sao, ta lưu trong `set` chứ không phải mảng.

Hướng thật của robot có thể khác "lên"? Không quan trọng: mọi hướng của ta đều **so với hướng ban đầu**, nên hình học vẫn nhất quán (chỉ là bản đồ bị xoay).

In [3]:
pos, d = (0, 0), 0
visited = {pos}
for step, action in enumerate(["move", "right", "move", "move", "left", "move"]):
    if action == "move":
        pos = (pos[0] + dr[d], pos[1] + dc[d]); visited.add(pos)
    elif action == "right":
        d = (d + 1) % 4
    else:
        d = (d + 3) % 4
    print(f"{action:<5} -> pos={pos}, hướng={names[d]}")
print("các ô đã đi:", sorted(visited))

move  -> pos=(-1, 0), hướng=lên
right -> pos=(-1, 0), hướng=phải
move  -> pos=(-1, 1), hướng=phải
move  -> pos=(-1, 2), hướng=phải
left  -> pos=(-1, 2), hướng=lên
move  -> pos=(-2, 2), hướng=lên
các ô đã đi: [(-2, 2), (-1, 0), (-1, 1), (-1, 2), (0, 0)]


### 3.3. DFS đệ quy và `set` visited

**DFS (Depth-First Search)**: đi sâu hết mức theo một nhánh, hết đường thì quay lui (backtrack) thử nhánh khác. Với lưới, DFS thăm mọi ô liên thông. `visited` (tập) đảm bảo mỗi ô chỉ thăm một lần.

Ví dụ DFS trên lưới **biết trước** (chưa có robot):

In [4]:
def dfs_known(grid, r, c, seen):
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] == 0 or (r, c) in seen:
        return
    seen.add((r, c))
    for k in range(4):
        dfs_known(grid, r + dr[k], c + dc[k], seen)

seen = set()
dfs_known(room1, 1, 3, seen)
print("số ô trống liên thông với (1,3):", len(seen))

số ô trống liên thông với (1,3): 30


### 3.4. Backtracking "vật lý": quay về ô trước

Trong DFS thường, "quay lui" là tự động (hàm return). Với robot, nó **đứng ở ô con** khi hàm return! Ta phải ra lệnh cho nó về:

```text
go_back:  turnRight, turnRight   (quay 180°)
          move                   (lùi về ô cha)
          turnRight, turnRight   (quay lại hướng cũ)
```

Sau `go_back`, robot ở ô cha và **cùng hướng** như trước khi tiến vào ô con. Điều này giúp vòng lặp ở ô cha tiếp tục đúng.

In [5]:
rb = RobotSim(room1, 1, 3)
print("ban đầu:", (rb.r, rb.c), "hướng", rb.d)
rb.turnRight(); rb.move()                    # sang phải 1 ô
print("sau khi tiến:", (rb.r, rb.c), "hướng", rb.d)
rb.turnRight(); rb.turnRight(); rb.move(); rb.turnRight(); rb.turnRight()
print("sau go_back:", (rb.r, rb.c), "hướng", rb.d)

ban đầu: (1, 3) hướng 0
sau khi tiến: (1, 4) hướng 1
sau go_back: (1, 3) hướng 1


### 3.5. Giới hạn đệ quy của Python

Python mặc định chỉ cho đệ quy sâu khoảng 1000 tầng (`sys.getrecursionlimit()`). Phòng `100 × 200 = 20 000` ô có thể tạo đường DFS sâu 20 000 → `RecursionError`. Hai cách xử lý: `sys.setrecursionlimit(...)` hoặc viết DFS **lặp** bằng stack (Cách 3).

In [6]:
import sys
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

giới hạn đệ quy hiện tại: 1000


## 4. Ý tưởng nền tảng / trực giác

Hãy tưởng tượng bạn bị bịt mắt trong một căn phòng, cầm một cuộn chỉ:
- Mỗi khi bước sang ô mới, bạn ghi nhớ ô đó (theo tọa độ tự đặt).
- Ở mỗi ô, bạn thử **lần lượt 4 hướng**. Hướng nào mở và dẫn tới ô **chưa thăm** → đi vào, khám phá hết rồi **lần theo cuộn chỉ** quay về.
- Khi đã thử đủ 4 hướng, ô này coi như xong, quay về ô trước đó.

Tính tay phòng nhỏ:

```text
room = [[1, 1],
        [1, 0]]     xuất phát (0,0) hướng lên
```

- Ở `(0,0)`: lau. Thử lên → biên → False. Quay phải (hướng phải): move → `(0,1)`.
  - Ở `(0,1)`: lau. Thử phải → biên; xuống → tường; trái → `(0,0)` đã thăm (không move); lên → biên. Xong → go_back về `(0,0)`.
- Ở `(0,0)` tiếp: thử xuống → `(1,0)`, lau, thử hết, go_back. Thử trái → biên. Xong.

Tổng: lau 3 ô = mọi ô trống. Đây chính là thuật toán.

In [7]:
small = [[1, 1], [1, 0]]
seen = set(); dfs_known(small, 0, 0, seen)
print("Các ô cần lau:", sorted(seen))

Các ô cần lau: [(0, 0), (0, 1), (1, 0)]


## 5. Cách 1 — Baseline: DFS "quay về hướng tuyệt đối"

**Ý tưởng.** Ở mỗi ô, với `target` lần lượt là `0, 1, 2, 3` (lên, phải, xuống, trái theo hệ tương đối), ta **xoay robot tới đúng hướng `target`** (gọi `turnRight` cho đến khi hướng hiện tại = `target`), rồi:
- Nếu ô phía trước chưa thăm và `move()` thành công → đệ quy; sau đó go_back.

Ta phải tự theo dõi biến `facing` (hướng hiện tại) vì robot không cho biết hướng.

Đây là cách "dễ nghĩ" nhất: tư duy theo hướng tuyệt đối như DFS trên lưới thường. Nhược điểm: phải tự quản lý một biến **trạng thái hướng toàn cục** `facing` ở mọi chỗ gọi lệnh quay — quên một chỗ là sai toàn bộ.

**Tham chiếu "brute force" để kiểm thử:** vì ta tự dựng phòng, ta biết tập ô cần lau bằng BFS trên bản đồ (hàm `reachable`). Mọi cách đều phải lau đúng tập đó.

In [8]:
from collections import deque

def reachable(room, row, col):
    """Tập ô trống liên thông với (row, col) — dùng làm đáp án chuẩn."""
    seen = {(row, col)}
    q = deque([(row, col)])
    while q:
        r, c = q.popleft()
        for k in range(4):
            nr, nc = r + dr[k], c + dc[k]
            if 0 <= nr < len(room) and 0 <= nc < len(room[0]) and room[nr][nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc)); q.append((nr, nc))
    return seen

def clean_absolute(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set()
    state = {"facing": 0}

    def face(target):
        while state["facing"] != target:
            robot.turnRight()
            state["facing"] = (state["facing"] + 1) % 4

    def dfs(r, c):
        visited.add((r, c))
        robot.clean()
        for target in range(4):
            nr, nc = r + DR[target], c + DC[target]
            if (nr, nc) in visited:
                continue
            face(target)
            if robot.move():
                dfs(nr, nc)
                # go back: quay 180, đi, quay 180 (cập nhật facing)
                face((target + 2) % 4)
                robot.move()
                face(target)

    dfs(0, 0)

rb = RobotSim(room1, 1, 3)
clean_absolute(rb)
print("lau", len(rb.cleaned), "ô; cần", len(reachable(room1, 1, 3)), "ô; moves =", rb.moves, "turns =", rb.turns)
assert rb.cleaned == reachable(room1, 1, 3)

lau 30 ô; cần 30 ô; moves = 104 turns = 220


**Độ phức tạp Cách 1.** Gọi `N` = số ô trống, `M` = số tường kề. Mỗi ô: tối đa 4 lần thử `move`, mỗi cạnh của cây DFS đi 2 lần (tới và về). Số lệnh quay mỗi ô bị chặn bởi một hằng số (xoay tới hướng cần thử, quay 180 khi lùi). → Time `O(N)`. Space `O(N)` cho `visited` và stack đệ quy.

**Nút thắt.** (1) Trạng thái hướng `facing` dùng chung giữa mọi tầng đệ quy — khó chứng minh, khó chuyển sang dạng lặp, dễ sai khi sửa code. (2) Đệ quy sâu có thể vượt giới hạn Python. Ta muốn một cách mà **mỗi tầng tự khôi phục hướng** để có bất biến đơn giản.

## 6. Bước đột phá

**Insight 1 — Thử hướng theo vòng xoắn (spiral).** Thay vì xoay tới hướng tuyệt đối, ở mỗi ô ta thử **hướng hiện tại trước**, rồi `turnRight` để thử hướng kế tiếp, và cứ thế 4 lần. Sau 4 lần `turnRight`, robot tự động quay về hướng ban đầu — không cần đếm/so hướng.

Hướng thứ `k` được thử là `(d + k) % 4`, với `d` là hướng khi vào ô. Ta chỉ cần truyền `d` vào đệ quy.

**Insight 2 — `go_back` khôi phục hướng.** Sau khi khám phá ô con theo hướng `nd`, `go_back` (quay 180, đi, quay 180) đưa robot về ô cha **với hướng `nd`**. Khi đó lệnh `turnRight` tiếp theo trong vòng lặp của ô cha đưa sang hướng `nd + 1` — đúng như kế hoạch.

**Bất biến (invariant):**
- Khi `dfs(r, c, d)` bắt đầu: robot đứng ở `(r, c)` (tọa độ tương đối), quay về hướng `d`.
- Khi `dfs(r, c, d)` kết thúc: robot vẫn ở `(r, c)`, hướng `d` (vì đã quay phải đúng 4 lần và mọi nhánh con đều go_back).

**Quy tắc:** **nếu** ô phía trước chưa thăm **và** `move()` trả `True` **thì** đệ quy vào đó, rồi go_back; **luôn** `turnRight` sau mỗi lần thử.

In [9]:
# Minh họa Insight 1: 4 lần turnRight đưa hướng về như cũ
d0 = 2
d = d0
tried = []
for k in range(4):
    tried.append((d0 + k) % 4)
    d = (d + 1) % 4
print("thử các hướng:", tried, "| hướng cuối = hướng đầu?", d == d0)

thử các hướng: [2, 3, 0, 1] | hướng cuối = hướng đầu? True


## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS xoắn ốc + backtracking (lời giải chuẩn)

In [10]:
def clean_spiral(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set()

    def go_back():
        robot.turnRight(); robot.turnRight()
        robot.move()
        robot.turnRight(); robot.turnRight()

    def dfs(r, c, d):
        visited.add((r, c))
        robot.clean()
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited and robot.move():
                dfs(nr, nc, nd)
                go_back()
            robot.turnRight()          # luôn quay sang hướng kế tiếp

    dfs(0, 0, 0)

rb = RobotSim(room1, 1, 3)
clean_spiral(rb)
print("lau", len(rb.cleaned), "ô; moves =", rb.moves, "turns =", rb.turns, "; về lại ô đầu?", (rb.r, rb.c) == (1, 3), "hướng đầu?", rb.d == 0)
assert rb.cleaned == reachable(room1, 1, 3)

lau 30 ô; moves = 104 turns = 236 ; về lại ô đầu? True hướng đầu? True


**Code nộp LeetCode (Cách 2):**

In [11]:
import sys

class SolutionSpiral:
    def cleanRoom(self, robot):
        sys.setrecursionlimit(100000)
        DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
        visited = set()

        def go_back():
            robot.turnRight(); robot.turnRight()
            robot.move()
            robot.turnRight(); robot.turnRight()

        def dfs(r, c, d):
            visited.add((r, c))
            robot.clean()
            for k in range(4):
                nd = (d + k) % 4
                nr, nc = r + DR[nd], c + DC[nd]
                if (nr, nc) not in visited and robot.move():
                    dfs(nr, nc, nd)
                    go_back()
                robot.turnRight()

        dfs(0, 0, 0)

rb = RobotSim(room1, 1, 3)
SolutionSpiral().cleanRoom(rb)
print(rb.cleaned == reachable(room1, 1, 3))

True


**Độ phức tạp Cách 2:** mỗi ô trống thăm đúng một lần; tại mỗi ô 4 lần thử + 4 lần quay; mỗi cạnh cây DFS tốn thêm 1 go_back (4 quay + 1 move). Time `O(N)` với `N` = số ô trống (hằng số nhỏ hơn Cách 1). Space `O(N)` (visited + stack đệ quy sâu tối đa `N`).

### 7.2. Cách 3 — DFS lặp với stack tường minh (không đệ quy)

Để không phụ thuộc giới hạn đệ quy, ta mô phỏng call stack. Mỗi phần tử stack là `[r, c, d, k]`: đang ở ô `(r, c)`, vào ô với hướng `d`, đã thử xong `k` hướng.

Vòng lặp:
- Nếu `k == 4` ở đỉnh stack: ô này xong → pop. Nếu còn ô cha: `go_back()` (robot về ô cha, hướng `nd` của cha), rồi `turnRight()` và tăng `k` của cha (hoàn tất lần thử đó).
- Ngược lại: tính hướng `nd = (d + k) % 4`, ô `(nr, nc)`. Nếu chưa thăm và `move()` thành công → thăm, lau, push `[nr, nc, nd, 0]` (chưa `turnRight`, vì lần thử của cha chưa xong). Nếu không → `turnRight()`, `k += 1`.

In [12]:
def clean_iterative(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]

    def go_back():
        robot.turnRight(); robot.turnRight()
        robot.move()
        robot.turnRight(); robot.turnRight()

    visited = {(0, 0)}
    robot.clean()
    stack = [[0, 0, 0, 0]]               # r, c, d, k
    while stack:
        top = stack[-1]
        r, c, d, k = top
        if k == 4:
            stack.pop()
            if stack:
                go_back()                # về ô cha, hướng = hướng đã dùng để vào ô con
                robot.turnRight()        # hoàn tất lần thử đó ở ô cha
                stack[-1][3] += 1
            continue
        nd = (d + k) % 4
        nr, nc = r + DR[nd], c + DC[nd]
        if (nr, nc) not in visited and robot.move():
            visited.add((nr, nc))
            robot.clean()
            stack.append([nr, nc, nd, 0])
        else:
            robot.turnRight()
            top[3] += 1

rb = RobotSim(room1, 1, 3)
clean_iterative(rb)
print("lau", len(rb.cleaned), "ô; moves =", rb.moves, "turns =", rb.turns, "; về ô đầu?", (rb.r, rb.c) == (1, 3))
assert rb.cleaned == reachable(room1, 1, 3)

lau 30 ô; moves = 104 turns = 236 ; về ô đầu? True


**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionIterative:
    def cleanRoom(self, robot):
        DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]

        def go_back():
            robot.turnRight(); robot.turnRight()
            robot.move()
            robot.turnRight(); robot.turnRight()

        visited = {(0, 0)}
        robot.clean()
        stack = [[0, 0, 0, 0]]
        while stack:
            r, c, d, k = stack[-1]
            if k == 4:
                stack.pop()
                if stack:
                    go_back()
                    robot.turnRight()
                    stack[-1][3] += 1
                continue
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited and robot.move():
                visited.add((nr, nc))
                robot.clean()
                stack.append([nr, nc, nd, 0])
            else:
                robot.turnRight()
                stack[-1][3] += 1

rb = RobotSim(room1, 1, 3)
SolutionIterative().cleanRoom(rb)
print(rb.cleaned == reachable(room1, 1, 3))

True


**Độ phức tạp Cách 3:** giống Cách 2 — Time `O(N)`, Space `O(N)` — nhưng không lo `RecursionError`. Thứ tự lệnh gửi cho robot **giống hệt** Cách 2 (ta sẽ kiểm tra ở mục 12).

## 8. TRACE từng bước

Trace **Cách 2** trên phòng nhỏ:

```text
room = [[1, 1],
        [1, 0]]    xuất phát (0,0), hướng lên
```

Bảng trạng thái (tọa độ tương đối trùng tọa độ thật vì xuất phát ở (0,0) hướng lên):

| bước | ô | hướng thử | ô đích | kết quả |
|---|---|---|---|---|
| 1 | (0,0) | lên | (-1,0) | move False → turnRight |
| 2 | (0,0) | phải | (0,1) | move True → dfs(0,1) |
| 3 | (0,1) | phải | (0,2) | False |
| 4 | (0,1) | xuống | (1,1) | tường, False |
| 5 | (0,1) | trái | (0,0) | đã thăm, không move |
| 6 | (0,1) | lên | (-1,1) | False → xong, go_back |
| 7 | (0,0) | xuống | (1,0) | True → dfs(1,0) ... go_back |
| 8 | (0,0) | trái | (0,-1) | False → xong |

In [14]:
def trace_spiral(room, row, col):
    robot = RobotSim(room, row, col, log=True)
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    names = ["lên", "phải", "xuống", "trái"]
    visited = set()

    def go_back():
        print("   go_back")
        robot.turnRight(); robot.turnRight(); robot.move(); robot.turnRight(); robot.turnRight()

    def dfs(r, c, d, depth):
        ind = "  " * depth
        visited.add((r, c))
        print(f"{ind}vào ô tương đối ({r},{c}), hướng {names[d]}")
        robot.clean()
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) in visited:
                print(f"{ind}  thử {names[nd]} -> ({nr},{nc}) đã thăm, bỏ qua")
            else:
                print(f"{ind}  thử {names[nd]} -> ({nr},{nc})")
                if robot.move():
                    dfs(nr, nc, nd, depth + 1)
                    go_back()
            robot.turnRight()
    dfs(0, 0, 0, 0)
    print("Đã lau:", sorted(robot.cleaned), "| moves:", robot.moves, "turns:", robot.turns)

trace_spiral([[1, 1], [1, 0]], 0, 0)

vào ô tương đối (0,0), hướng lên
   clean (0,0)
  thử lên -> (-1,0)
   move -> bị chặn
  thử phải -> (0,1)
   move -> (0,1)
  vào ô tương đối (0,1), hướng phải
   clean (0,1)
    thử phải -> (0,2)
   move -> bị chặn
    thử xuống -> (1,1)
   move -> bị chặn
    thử trái -> (0,0) đã thăm, bỏ qua
    thử lên -> (-1,1)
   move -> bị chặn
   go_back
   move -> (0,0)
  thử xuống -> (1,0)
   move -> (1,0)
  vào ô tương đối (1,0), hướng xuống
   clean (1,0)
    thử xuống -> (2,0)
   move -> bị chặn
    thử trái -> (1,-1)
   move -> bị chặn
    thử lên -> (0,0) đã thăm, bỏ qua
    thử phải -> (1,1)
   move -> bị chặn
   go_back
   move -> (0,0)
  thử trái -> (0,-1)
   move -> bị chặn
Đã lau: [(0, 0), (0, 1), (1, 0)] | moves: 12 turns: 20


Trace **Cách 3** (stack) trên cùng phòng — in nội dung stack sau mỗi vòng lặp:

In [15]:
def trace_iterative(room, row, col):
    robot = RobotSim(room, row, col)
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    def go_back():
        robot.turnRight(); robot.turnRight(); robot.move(); robot.turnRight(); robot.turnRight()
    visited = {(0, 0)}; robot.clean()
    stack = [[0, 0, 0, 0]]
    it = 0
    while stack:
        it += 1
        r, c, d, k = stack[-1]
        if k == 4:
            stack.pop()
            act = "pop"
            if stack:
                go_back(); robot.turnRight(); stack[-1][3] += 1
                act += " + go_back"
        else:
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited and robot.move():
                visited.add((nr, nc)); robot.clean(); stack.append([nr, nc, nd, 0])
                act = f"tiến tới ({nr},{nc})"
            else:
                robot.turnRight(); stack[-1][3] += 1
                act = f"thử ({nr},{nc}) thất bại"
        print(f"vòng {it:>2}: {act:<26} stack={stack}  vị trí thật={(robot.r, robot.c)}")

trace_iterative([[1, 1], [1, 0]], 0, 0)

vòng  1: thử (-1,0) thất bại        stack=[[0, 0, 0, 1]]  vị trí thật=(0, 0)
vòng  2: tiến tới (0,1)             stack=[[0, 0, 0, 1], [0, 1, 1, 0]]  vị trí thật=(0, 1)
vòng  3: thử (0,2) thất bại         stack=[[0, 0, 0, 1], [0, 1, 1, 1]]  vị trí thật=(0, 1)
vòng  4: thử (1,1) thất bại         stack=[[0, 0, 0, 1], [0, 1, 1, 2]]  vị trí thật=(0, 1)
vòng  5: thử (0,0) thất bại         stack=[[0, 0, 0, 1], [0, 1, 1, 3]]  vị trí thật=(0, 1)
vòng  6: thử (-1,1) thất bại        stack=[[0, 0, 0, 1], [0, 1, 1, 4]]  vị trí thật=(0, 1)
vòng  7: pop + go_back              stack=[[0, 0, 0, 2]]  vị trí thật=(0, 0)
vòng  8: tiến tới (1,0)             stack=[[0, 0, 0, 2], [1, 0, 2, 0]]  vị trí thật=(1, 0)
vòng  9: thử (2,0) thất bại         stack=[[0, 0, 0, 2], [1, 0, 2, 1]]  vị trí thật=(1, 0)
vòng 10: thử (1,-1) thất bại        stack=[[0, 0, 0, 2], [1, 0, 2, 2]]  vị trí thật=(1, 0)
vòng 11: thử (0,0) thất bại         stack=[[0, 0, 0, 2], [1, 0, 2, 3]]  vị trí thật=(1, 0)
vòng 12: thử (1,1) thất bại

## 9. Code nộp LeetCode

Bản nộp chính: Cách 2 (DFS xoắn ốc), có nâng giới hạn đệ quy cho phòng lớn.

In [16]:
import sys

class Solution:
    def cleanRoom(self, robot):
        """
        :type robot: Robot
        :rtype: None
        """
        sys.setrecursionlimit(100000)
        DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]     # lên, phải, xuống, trái (chiều kim đồng hồ)
        visited = set()

        def go_back():
            robot.turnRight(); robot.turnRight()
            robot.move()
            robot.turnRight(); robot.turnRight()

        def dfs(r, c, d):
            visited.add((r, c))
            robot.clean()
            for k in range(4):
                nd = (d + k) % 4
                nr, nc = r + DIRS[nd][0], c + DIRS[nd][1]
                if (nr, nc) not in visited and robot.move():
                    dfs(nr, nc, nd)
                    go_back()
                robot.turnRight()

        dfs(0, 0, 0)

for room, r0, c0 in [(room1, 1, 3), ([[1]], 0, 0), ([[1, 1], [1, 0]], 0, 0)]:
    rb = RobotSim(room, r0, c0)
    Solution().cleanRoom(rb)
    assert rb.cleaned == reachable(room, r0, c0)
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (bất biến vị trí/hướng).** Mỗi lời gọi `dfs(r, c, d)` bắt đầu và kết thúc với robot ở ô thật tương ứng `(r, c)` và hướng thật tương ứng `d`.
- Trong vòng lặp, sau lần thử thứ `k`, robot ở `(r, c)` hướng `d + k + 1`: nếu `move()` thất bại, robot đứng yên, rồi `turnRight`; nếu thành công, theo quy nạp `dfs` con trả robot về ô con hướng `nd`, `go_back` (quay 180, đi 1 ô, quay 180) đưa về `(r, c)` hướng `nd = d + k`, rồi `turnRight`.
- Sau 4 lần: hướng `d + 4 ≡ d`.

**Bổ đề 2 (tọa độ tương đối nhất quán).** Vì hướng tương đối luôn bằng hướng thật trừ một hằng số (hướng ban đầu), mỗi ô thật ứng với đúng một tọa độ tương đối. Do đó `visited` theo tọa độ tương đối phản ánh đúng các ô thật đã thăm.

**Bổ đề 3 (thăm đủ).** Giả sử có ô trống liên thông `X` không được lau. Lấy đường đi từ ô xuất phát tới `X`; có một cặp ô kề `(A, B)` trên đường với `A` được thăm, `B` không. Khi `dfs` ở `A` thử hướng tới `B`: `B` chưa thăm (nó không bao giờ được thăm) và trống nên `move()` trả `True` → `B` được thăm. Mâu thuẫn.

**Không thăm hai lần:** chỉ vào ô khi chưa có trong `visited`, và thêm vào `visited` ngay khi vào.

## 11. Phân tích Time/Space

Gọi `N` = số ô trống liên thông, `W` = số lần đụng tường/đã thăm.

| Đại lượng | Cách 1 (tuyệt đối) | Cách 2 (xoắn ốc) | Cách 3 (stack) |
|---|---|---|---|
| Số `move` | `≤ 4N + (N-1)` | `≤ 4N + (N-1)` | như Cách 2 |
| Số quay | tùy hình dạng phòng (bỏ qua hướng đã thăm) | đúng `4N + 4(N-1)` | như Cách 2 |
| Time | `O(N)` | `O(N)` | `O(N)` |
| Space | `O(N)` | `O(N)` | `O(N)` không đệ quy |

Đo trên phòng ngẫu nhiên lớn:

In [17]:
import random
random.seed(489)
R, C = 60, 120
big_room = [[1 if random.random() < 0.75 else 0 for _ in range(C)] for _ in range(R)]
big_room[30][60] = 1
sys.setrecursionlimit(100000)
for name, f in [("absolute", clean_absolute), ("spiral", clean_spiral), ("iterative", clean_iterative)]:
    rb = RobotSim(big_room, 30, 60)
    f(rb)
    ok = rb.cleaned == reachable(big_room, 30, 60)
    print(f"{name:<10} đúng={ok} | ô={len(rb.cleaned)} | moves={rb.moves} | turns={rb.turns}")

absolute   đúng=True | ô=5432 | moves=16210 | turns=36492
spiral     đúng=True | ô=5432 | moves=16210 | turns=43452
iterative  đúng=True | ô=5432 | moves=16210 | turns=43452


**Nhận xét từ số liệu:** cả ba cách gửi **cùng số lệnh `move`**. Cách 1 lại dùng **ít lệnh quay hơn** vì nó không cần quay qua những hướng dẫn tới ô đã thăm. Đề bài không yêu cầu tối thiểu số lệnh, nên Cách 2 vẫn là lựa chọn phổ biến: code ngắn, mỗi tầng đệ quy tự khôi phục hướng (bất biến rõ ràng), và dễ chuyển sang dạng lặp (Cách 3). Nếu một biến thể yêu cầu tiết kiệm năng lượng quay, Cách 1 đáng cân nhắc.

## 12. Bộ test

Kiểm thử gồm: ví dụ đề, phòng 1 ô, phòng một hàng, phòng có ô trống **không** liên thông (robot không được — và không thể — tới), hướng xuất phát thật **khác "lên"** (thuật toán vẫn đúng nhờ tọa độ tương đối), và phòng ngẫu nhiên có seed. "Brute force" ở đây là `reachable` (BFS trên bản đồ).

In [18]:
def run(f, room, r, c, facing=0):
    rb = RobotSim(room, r, c, facing=facing)
    f(rb)
    return rb

methods = {
    "absolute": clean_absolute,
    "spiral": clean_spiral,
    "iterative": clean_iterative,
    "SolSpiral": SolutionSpiral().cleanRoom,
    "SolIter": SolutionIterative().cleanRoom,
    "Solution": Solution().cleanRoom,
}
fixed = [
    (room1, 1, 3),
    ([[1]], 0, 0),
    ([[1, 1, 1, 1, 1]], 0, 2),
    ([[1], [1], [1]], 2, 0),
    ([[1, 0, 1], [1, 0, 1], [1, 1, 1]], 0, 0),       # chữ U
    ([[1, 0, 1], [0, 0, 0], [1, 0, 1]], 0, 0),       # các ô bị cô lập
    ([[1] * 6 for _ in range(5)], 2, 3),             # phòng trống hoàn toàn
]
for room, r, c in fixed:
    exp = reachable(room, r, c)
    for facing in range(4):
        for name, f in methods.items():
            rb = run(f, room, r, c, facing)
            assert rb.cleaned == exp, (name, room, facing)
print("Test cố định (mọi hướng xuất phát): OK")

Test cố định (mọi hướng xuất phát): OK


In [19]:
random.seed(2026)
for trial in range(300):
    R, C = random.randint(1, 12), random.randint(1, 12)
    p = random.choice([0.4, 0.6, 0.8, 1.0])
    room = [[1 if random.random() < p else 0 for _ in range(C)] for _ in range(R)]
    r, c = random.randrange(R), random.randrange(C)
    room[r][c] = 1
    facing = random.randrange(4)
    exp = reachable(room, r, c)
    logs = []
    for name, f in methods.items():
        rb = run(f, room, r, c, facing)
        assert rb.cleaned == exp, (name, trial)
        if name in ("spiral", "iterative"):
            logs.append((rb.moves, rb.turns, rb.r, rb.c, rb.d))
    # Cách 2 và 3 gửi cùng số lệnh và kết thúc ở ô xuất phát, đúng hướng ban đầu
    assert logs[0] == logs[1] and logs[0][2:] == (r, c, facing)
print("300 phòng ngẫu nhiên (seed 2026): mọi cách lau đúng tập ô liên thông")

300 phòng ngẫu nhiên (seed 2026): mọi cách lau đúng tập ô liên thông


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên go_back

**Code sai:**

```python
if (nr, nc) not in visited and robot.move():
    dfs(nr, nc, nd)          # robot vẫn đứng ở ô con!
robot.turnRight()
```

**Vì sao sai:** sau khi đệ quy xong, robot ở ô con nhưng code tưởng nó ở ô cha → tọa độ tương đối lệch, có ô bị bỏ sót hoặc "lau nhầm".

In [20]:
def clean_no_goback(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set()
    def dfs(r, c, d):
        visited.add((r, c)); robot.clean()
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited and robot.move():
                dfs(nr, nc, nd)
            robot.turnRight()
    dfs(0, 0, 0)

class CappedRobot(RobotSim):
    """RobotSim nhưng dừng (ném lỗi) nếu dùng quá nhiều lệnh — để demo code sai mà không treo máy."""
    CAP = 3000
    def move(self):
        if self.moves + self.turns > self.CAP:
            raise RuntimeError("quá nhiều lệnh")
        return super().move()

def run_capped(f, room, r, c):
    rb = CappedRobot(room, r, c)
    try:
        f(rb)
        status = "kết thúc"
    except (RuntimeError, RecursionError):
        status = "KHÔNG dừng (đi lang thang vô hạn)"
    return rb, status

room_u = [[1, 0, 1], [1, 0, 1], [1, 1, 1]]
rb, st = run_capped(clean_no_goback, room_u, 0, 0)
print("quên go_back:", st, "| lau", len(rb.cleaned), "/", len(reachable(room_u, 0, 0)), "ô")
rb = run(clean_spiral, room_u, 0, 0)
print("đúng        : lau", len(rb.cleaned), "/", len(reachable(room_u, 0, 0)), "ô, robot về", (rb.r, rb.c))

quên go_back: KHÔNG dừng (đi lang thang vô hạn) | lau 7 / 7 ô
đúng        : lau 7 / 7 ô, robot về (0, 0)


**Code đúng:** gọi `go_back()` ngay sau `dfs(...)` con.

### Lỗi 2 — `go_back` thiếu bước quay lại hướng cũ

**Code sai:**

```python
def go_back():
    robot.turnRight(); robot.turnRight()
    robot.move()                 # quên quay 180 lần nữa!
```

**Vì sao sai:** sau `go_back`, robot về đúng ô cha nhưng đang nhìn **ngược** hướng mà code nghĩ. Lệnh `turnRight` tiếp theo làm lệch toàn bộ hệ tọa độ tương đối → robot đi lang thang, có thể không bao giờ dừng hoặc bỏ sót ô.

In [21]:
def clean_bad_goback(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set()
    def go_back():
        robot.turnRight(); robot.turnRight(); robot.move()      # thiếu 2 lần quay
    def dfs(r, c, d):
        visited.add((r, c)); robot.clean()
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited and robot.move():
                dfs(nr, nc, nd); go_back()
            robot.turnRight()
    dfs(0, 0, 0)

random.seed(3)
bad = 0
for _ in range(100):
    room = [[1 if random.random() < 0.7 else 0 for _ in range(6)] for _ in range(6)]
    room[0][0] = 1
    rb, st = run_capped(clean_bad_goback, room, 0, 0)
    if st != "kết thúc" or rb.cleaned != reachable(room, 0, 0):
        bad += 1
print("Số phòng bị lỗi khi go_back thiếu quay:", bad, "/ 100")

Số phòng bị lỗi khi go_back thiếu quay: 75 / 100


**Code đúng:** `turnRight ×2, move, turnRight ×2`.

> **Ghi chú thú vị:** nếu bạn liệt kê `DIRS` **ngược chiều kim đồng hồ** nhưng vẫn dùng `turnRight`, code *vẫn đúng*! Bản đồ trong đầu chỉ bị **lật gương** một cách nhất quán (một song ánh giữa ô thật và tọa độ tương đối), nên `visited` vẫn phản ánh đúng. Điều quan trọng là **nhất quán**, không phải chiều cụ thể. Kiểm chứng:

In [22]:
def clean_mirror_dirs(robot):
    DIRS = [(-1, 0), (0, -1), (1, 0), (0, 1)]       # ngược chiều kim đồng hồ, vẫn dùng turnRight
    visited = set()
    def go_back():
        robot.turnRight(); robot.turnRight(); robot.move(); robot.turnRight(); robot.turnRight()
    def dfs(r, c, d):
        visited.add((r, c)); robot.clean()
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DIRS[nd][0], c + DIRS[nd][1]
            if (nr, nc) not in visited and robot.move():
                dfs(nr, nc, nd); go_back()
            robot.turnRight()
    dfs(0, 0, 0)

random.seed(4)
for _ in range(200):
    room = [[1 if random.random() < 0.7 else 0 for _ in range(6)] for _ in range(6)]
    room[0][0] = 1
    assert run(clean_mirror_dirs, room, 0, 0).cleaned == reachable(room, 0, 0)
print("DIRS lật gương vẫn lau đúng 200/200 phòng")

DIRS lật gương vẫn lau đúng 200/200 phòng


### Lỗi 3 — Gọi `move()` trước khi kiểm tra `visited`

**Code sai:**

```python
if robot.move() and (nr, nc) not in visited:   # thứ tự điều kiện ngược!
    dfs(nr, nc, nd); go_back()
```

**Vì sao sai:** nếu ô phía trước trống nhưng **đã thăm**, `move()` vẫn chạy và robot **đã bước sang đó**, nhưng code không đệ quy cũng không lùi lại → vị trí thật lệch khỏi tọa độ trong đầu, mọi bước sau sai.

**Code đúng:** `if (nr, nc) not in visited and robot.move():` — toán tử `and` của Python **ngắt mạch** (short-circuit): vế trái `False` thì vế phải (`move`) không được gọi.

In [23]:
def clean_wrong_order(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set()
    def go_back():
        robot.turnRight(); robot.turnRight(); robot.move(); robot.turnRight(); robot.turnRight()
    def dfs(r, c, d):
        visited.add((r, c)); robot.clean()
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if robot.move() and (nr, nc) not in visited:      # SAI thứ tự
                dfs(nr, nc, nd); go_back()
            robot.turnRight()
    dfs(0, 0, 0)

full = [[1] * 4 for _ in range(4)]
rb, st = run_capped(clean_wrong_order, full, 0, 0)
print("thứ tự sai:", st, "| vị trí cuối", (rb.r, rb.c), "| lau", len(rb.cleaned), "ô")
rb = run(clean_spiral, full, 0, 0)
print("đúng      : vị trí cuối", (rb.r, rb.c), "| lau", len(rb.cleaned), "ô")

thứ tự sai: KHÔNG dừng (đi lang thang vô hạn) | vị trí cuối (0, 1) | lau 16 ô
đúng      : vị trí cuối (0, 0) | lau 16 ô


### Lỗi 4 — `RecursionError` với phòng lớn

Phòng `100 × 200` rỗng → đường DFS sâu cỡ `20 000`. Không `sys.setrecursionlimit` sẽ crash.

In [24]:
corridor = [[1] * 200 for _ in range(10)]
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    run(clean_spiral, corridor, 0, 0)
    print("không lỗi")
except RecursionError:
    print("RecursionError với giới hạn 1000!")
finally:
    sys.setrecursionlimit(max(old, 100000))
rb = run(clean_iterative, corridor, 0, 0)
print("Cách 3 (stack) lau", len(rb.cleaned), "ô không cần đệ quy")

RecursionError với giới hạn 1000!
Cách 3 (stack) lau 2000 ô không cần đệ quy


**Code đúng:** `sys.setrecursionlimit(100000)` hoặc dùng Cách 3.

## 14. Mẹo & nhận dạng pattern

- **Bài interactive** + "không biết bản đồ" → **tọa độ tương đối** + `set` visited.
- Robot/agent di chuyển vật lý → **DFS + backtracking có hoàn tác hành động** (undo): mọi thứ làm khi đi xuống phải làm ngược khi đi lên.
- Khi chỉ có `turnRight` (hoặc chỉ `turnLeft`), thử hướng theo **vòng xoắn**: thử hướng hiện tại, quay, thử tiếp — 4 lần quay là về chỗ cũ.
- Tự dựng **lớp mô phỏng** có đếm lệnh và ghi log để kiểm thử — áp dụng cho mọi bài interactive (Guess Number, Find in Mountain Array, ...).
- DFS trên đồ thị có thể sâu cỡ `10^4` → nghĩ tới giới hạn đệ quy của Python.

## 15. Biến thể

1. **Robot báo cáo bản đồ:** trả về tập ô trống (tọa độ tương đối) — chính là `visited`.
2. **Đếm số ô trống / diện tích phòng** mà không lau.
3. **Tìm đường ngắn nhất tới ô đích** trong phòng ẩn: khám phá bằng DFS trước, sau đó BFS trên bản đồ đã biết (giống LC 1810 Minimum Path Cost in a Hidden Grid, LC 1778 Shortest Path in a Hidden Grid).
4. **Robot chỉ có `turnLeft`:** đổi thứ tự hướng thành ngược chiều kim đồng hồ.

Demo biến thể 1 + 3: khám phá bản đồ rồi BFS tìm khoảng cách xa nhất từ ô xuất phát.

In [25]:
def explore_map(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set()
    def go_back():
        robot.turnRight(); robot.turnRight(); robot.move(); robot.turnRight(); robot.turnRight()
    def dfs(r, c, d):
        visited.add((r, c))
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited and robot.move():
                dfs(nr, nc, nd); go_back()
            robot.turnRight()
    dfs(0, 0, 0)
    return visited

def farthest(cells):
    dist = {(0, 0): 0}; q = deque([(0, 0)])
    while q:
        r, c = q.popleft()
        for k in range(4):
            nb = (r + dr[k], c + dc[k])
            if nb in cells and nb not in dist:
                dist[nb] = dist[(r, c)] + 1; q.append(nb)
    return max(dist.values())

cells = explore_map(RobotSim(room1, 1, 3))
print("Số ô khám phá:", len(cells), "| khoảng cách xa nhất từ ô xuất phát:", farthest(cells))
assert len(cells) == len(reachable(room1, 1, 3))

Số ô khám phá: 30 | khoảng cách xa nhất từ ô xuất phát: 7


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Tham chiếu BFS | Biết bản đồ, BFS | `O(mn)` | `O(mn)` | Chỉ để kiểm thử (không hợp lệ khi nộp) |
| 1. DFS hướng tuyệt đối | Xoay tới từng hướng 0..3, tự giữ `facing` | `O(N)` | `O(N)` | Dễ nghĩ; muốn tiết kiệm lệnh quay |
| 2. DFS xoắn ốc | Thử hướng hiện tại, `turnRight`, go_back | `O(N)` | `O(N)` | Lời giải chuẩn, ngắn |
| 3. DFS lặp (stack) | Mô phỏng call stack `[r,c,d,k]` | `O(N)` | `O(N)` | Phòng lớn, tránh RecursionError |

## 17. Bài tập

**Bài 1 (đoán output).** Robot ở hướng `1` (phải), gọi `turnLeft` 3 lần. Hướng mới là gì?

**Bài 2 (tính tay).** Phòng `[[1,1,1]]`, xuất phát `(0,0)` hướng lên. Với Cách 2, robot gọi `move()` bao nhiêu lần (cả thành công lẫn thất bại)?

**Bài 3 (điền TODO).** Hoàn thiện `go_back_todo(robot)`.

In [26]:
def go_back_todo(robot):
    # TODO: quay 180 độ
    # TODO: đi 1 bước
    # TODO: quay 180 độ để về hướng cũ
    raise NotImplementedError

# Không gọi khi chạy notebook.

**Bài 4 (tự viết lại).** Viết lại `Solution` dùng **chỉ `turnLeft`** (không dùng `turnRight`).

**Bài 5 (mini challenge).** Viết `room_area_and_perimeter(robot)` trả về `(số ô trống, số cạnh giáp tường/biên)` của vùng robot đi được.

In [27]:
def room_area_and_perimeter_todo(robot):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: `turnLeft` = `d + 3 (mod 4)`.
- Mức 2: Bài 2: mỗi ô thử ≤ 4 hướng, nhưng hướng về ô đã thăm không gọi `move`; mỗi go_back gọi 1 `move`.
- Mức 3: Đếm bằng `RobotSim.moves`.

**Bài 3**
- Mức 1: 2 lệnh quay cùng chiều = 180°.
- Mức 2: Quay 180 → move → quay 180.
- Mức 3: `turnRight(); turnRight(); move(); turnRight(); turnRight()`.

**Bài 4**
- Mức 1: `turnLeft` = quay ngược chiều kim đồng hồ.
- Mức 2: Liệt kê hướng ngược chiều kim đồng hồ: lên, trái, xuống, phải.
- Mức 3: Trong go_back dùng `turnLeft` 2 lần.

**Bài 5**
- Mức 1: Diện tích = `len(visited)`.
- Mức 2: Chu vi = số lần `move()` thất bại (đụng tường/biên).
- Mức 3: Cẩn thận: mỗi cạnh tường chỉ bị thử **một lần** vì mỗi ô thử mỗi hướng đúng một lần.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert (1 + 3 * 3) % 4 == 2          # xuống
# Bài 2
rb = run(clean_spiral, [[1, 1, 1]], 0, 0)
print("Bài 2: số lần move =", rb.moves)

# Bài 3
def go_back_sol(robot):
    robot.turnRight(); robot.turnRight()
    robot.move()
    robot.turnRight(); robot.turnRight()
rb = RobotSim([[1, 1]], 0, 0, facing=1); rb.move(); go_back_sol(rb)
assert (rb.r, rb.c, rb.d) == (0, 0, 1)

# Bài 4
class SolutionLeft:
    def cleanRoom(self, robot):
        DIRS = [(-1, 0), (0, -1), (1, 0), (0, 1)]       # lên, trái, xuống, phải (ngược chiều kim đồng hồ)
        visited = set()
        def go_back():
            robot.turnLeft(); robot.turnLeft(); robot.move(); robot.turnLeft(); robot.turnLeft()
        def dfs(r, c, d):
            visited.add((r, c)); robot.clean()
            for k in range(4):
                nd = (d + k) % 4
                nr, nc = r + DIRS[nd][0], c + DIRS[nd][1]
                if (nr, nc) not in visited and robot.move():
                    dfs(nr, nc, nd); go_back()
                robot.turnLeft()
        dfs(0, 0, 0)

random.seed(17)
for _ in range(200):
    R, C = random.randint(1, 9), random.randint(1, 9)
    room = [[1 if random.random() < 0.65 else 0 for _ in range(C)] for _ in range(R)]
    r, c = random.randrange(R), random.randrange(C); room[r][c] = 1
    assert run(SolutionLeft().cleanRoom, room, r, c, random.randrange(4)).cleaned == reachable(room, r, c)
print("Bài 1–4 OK")

Bài 2: số lần move = 12
Bài 1–4 OK


In [29]:
# Bài 5
def room_area_and_perimeter(robot):
    DR, DC = [-1, 0, 1, 0], [0, 1, 0, -1]
    visited = set(); walls = [0]
    def go_back():
        robot.turnRight(); robot.turnRight(); robot.move(); robot.turnRight(); robot.turnRight()
    def dfs(r, c, d):
        visited.add((r, c))
        for k in range(4):
            nd = (d + k) % 4
            nr, nc = r + DR[nd], c + DC[nd]
            if (nr, nc) not in visited:
                if robot.move():
                    dfs(nr, nc, nd); go_back()
                else:
                    walls[0] += 1
            robot.turnRight()
    dfs(0, 0, 0)
    return len(visited), walls[0]

def area_perimeter_ref(room, r, c):
    cells = reachable(room, r, c)
    per = sum(1 for (x, y) in cells for k in range(4) if (x + dr[k], y + dc[k]) not in cells)
    return len(cells), per

random.seed(99)
for _ in range(200):
    R, C = random.randint(1, 9), random.randint(1, 9)
    room = [[1 if random.random() < 0.65 else 0 for _ in range(C)] for _ in range(R)]
    r, c = random.randrange(R), random.randrange(C); room[r][c] = 1
    assert room_area_and_perimeter(RobotSim(room, r, c)) == area_perimeter_ref(room, r, c)
print("Bài 5 OK:", room_area_and_perimeter(RobotSim(room1, 1, 3)))

Bài 5 OK: (30, 46)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DFS robot xoắn ốc:**

```python
DIRS = [(-1,0), (0,1), (1,0), (0,-1)]          # chiều kim đồng hồ <-> turnRight
visited = set()
def go_back(): R(); R(); robot.move(); R(); R()  # R = robot.turnRight
def dfs(r, c, d):
    visited.add((r, c)); robot.clean()
    for k in range(4):
        nd = (d + k) % 4; nr, nc = r + DIRS[nd][0], c + DIRS[nd][1]
        if (nr, nc) not in visited and robot.move():
            dfs(nr, nc, nd); go_back()
        robot.turnRight()
dfs(0, 0, 0)
```

**Checklist trước khi submit:**

- [ ] Thứ tự `DIRS` nhất quán với chiều quay (quy ước: `turnRight` ↔ chiều kim đồng hồ).
- [ ] Sau mỗi `dfs` con có `go_back()`; sau mỗi lần thử có `turnRight()`.
- [ ] `visited` dùng tọa độ **tương đối**, thêm ngay khi vào ô; `clean()` ngay khi vào ô.
- [ ] Kiểm tra `visited` **trước** khi gọi `move()` (tránh đi vào ô đã thăm rồi phải lùi).
- [ ] Nâng `sys.setrecursionlimit` hoặc dùng stack lặp.
- [ ] Đã kiểm thử bằng lớp mô phỏng với nhiều hướng xuất phát và phòng ngẫu nhiên.